In [ ]:
import pandas as pd

# First we load employee census information
df = pd.read_csv("RAW_DATA")

In [3]:
print("Dataset preview:")

# Then we check if we can see some null data or any missing information by first
df.head()

Dataset preview:


,ID,Birth_Date_(MM/DD/YYYY),Hire_Date,Termination_Date,Rehire_Date,Gross_compensation,Plan_eligible_comp,Employee_deduction_1_Pre-tax_amount,Employee_deduction_2_Roth_Deferral,Employer_Match,Hours_worked_2022,Employee_Status,Entry_date
0,Employee 1,5/14/44,1/7/20,0,0,"$82,460.35","$82,460.35","$17,219.53",$0.00,13948.47,2080,Full-Time,3/1/20
1,Employee 2,12/18/31,1/19/19,0,0,"$36,328.55","$36,328.55","$1,018.17",$0.00,294.50,1500,Part-Time,3/1/19
2,Employee 3,8/28/48,1/1/20,0,0,"$319,773.45","$319,773.45","$8,716.71",$0.00,7472.55,2080,Full-Time,2/1/20
3,Employee 4,10/20/72,11/18/13,0,0,"$60,120.66","$60,120.66",$0.00,$0.00,0.00,2148,Full-Time,1/1/14
4,Employee 5,4/3/91,6/7/17,2/20/23,0,"$71,936.25","$71,936.25","$3,925.84",$0.00,3515.56,2080,Full-Time,8/1/17


In [4]:

print("\nDataset info:")

# After using .info we check if there's any null value and the Dtype
df.info()


Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 13 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ID                                   82 non-null     object 
 1   Birth_Date_(MM/DD/YYYY)              82 non-null     object 
 2   Hire_Date                            82 non-null     object 
 3   Termination_Date                     82 non-null     object 
 4   Rehire_Date                          82 non-null     int64  
 5   Gross_compensation                   82 non-null     object 
 6   Plan_eligible_comp                   82 non-null     object 
 7   Employee_deduction_1_Pre-tax_amount  82 non-null     object 
 8   Employee_deduction_2_Roth_Deferral   82 non-null     object 
 9   Employer_Match                       82 non-null     float64
 10  Hours_worked_2022                    82 non-null     int64  
 11  Employee_Status    

In [5]:
print("\nMissing values:")

# After identify the exact column we proceed to get more context of that exact null
df.isnull().sum()


Missing values:


,0
ID,0
Birth_Date_(MM/DD/YYYY),0
Hire_Date,0
Termination_Date,0
Rehire_Date,0
Gross_compensation,0
Plan_eligible_comp,0
Employee_deduction_1_Pre-tax_amount,0
Employee_deduction_2_Roth_Deferral,0
Employer_Match,0


In [6]:
# Checking the dataset I noticed that the format is not fully readable

# So, I rename the date only if it has not been changed yet.
if 'Birth_Date_(MM/DD/YYYY)' in df.columns:
    df.rename(columns={'Birth_Date_(MM/DD/YYYY)': 'Birth_Date'}, inplace=True)

In [7]:
print("\nDate Cleaning")

# Now we're gonna fix the date time
date_columns = [
    'Birth_Date',
    'Hire_Date',
    'Termination_Date',
    'Rehire_Date',
    'Entry_date'
]

def fix_future_dates(series, max_valid_year=2022):
    dates = pd.to_datetime(series, errors='coerce', format='%m/%d/%y')

    # the lamba function is made to extract 100 years
    # only if the valid_year is below 2022
    fixed_dates = dates.apply(
        lambda x: x.replace(year=x.year - 100)
        if pd.notnull(x) and x.year > max_valid_year
        else x
    )

    return fixed_dates


# Fix future years only for Birth_Date
df['Birth_Date'] = fix_future_dates(df['Birth_Date'])

# Convert remaining date columns normally
for col in ['Hire_Date', 'Termination_Date', 'Rehire_Date', 'Entry_date']:
    df[col] = pd.to_datetime(
        df[col].replace('0', pd.NaT),
        errors='coerce'
    )


Date Cleaning


/tmp/ipykernel_3931/2422222822.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_3931/2422222822.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_3931/2422222822.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(


In [8]:
print("\nEligibility Filtering")

# Check missing entry dates
missing_entries = df[df['Entry_date'].isna()]
print(f"Employees with missing/invalid Entry Date: {len(missing_entries)}")

# -------------------
# Eligibility Rules
# -------------------

# Rule 1: Entry before Jan 1, 2023
eligible_entry = df['Entry_date'] < pd.Timestamp('2023-01-01')

# Rule 2: Must be employed on last day of 2022 and entry date before January 1, 2023
last_day_rule = (
    df['Termination_Date'].isna() |
    (df['Termination_Date'] >= pd.Timestamp('2022-12-31'))
)

# Rule 3: Minimum hours
hours_requirement = df['Hours_worked_2022'] >= 1000

# Rule 4: Full-time only
status_filter = (
    df['Employee_Status']
    .str.lower()
    .str.strip()
    .str.replace(" ", "-", regex=False)
    == 'full-time'
)

# Final filter
final_eligibility = (
    eligible_entry &
    last_day_rule &
    hours_requirement &
    status_filter
)

# Create eligible dataset
df_eligible_employees = df[final_eligibility].copy()

# -------------------
# Validation
# -------------------

print(f"Total employees: {len(df)}")
print(f"Eligible employees: {len(df_eligible_employees)}")
print(f"Excluded employees: {len(df) - len(df_eligible_employees)}")

print("\nBreakdown of exclusions:")
print(f"Failed entry rule: {(~eligible_entry).sum()}")
print(f"Failed last day rule: {(~last_day_rule).sum()}")
print(f"Failed hours rule: {(~hours_requirement).sum()}")
print(f"Failed status rule: {(~status_filter).sum()}")


Eligibility Filtering
Employees with missing/invalid Entry Date: 1
Total employees: 82
Eligible employees: 54
Excluded employees: 28

Breakdown of exclusions:
Failed entry rule: 5
Failed last day rule: 10
Failed hours rule: 19
Failed status rule: 13


In [9]:
# After checking again the .info I see that there's some object instead of currency
print("\nConvert monetary columns")

money_columns = [
    'Gross_compensation',
    'Plan_eligible_comp',
    'Employee_deduction_1_Pre-tax_amount',
    'Employee_deduction_2_Roth_Deferral',
    'Employer_Match'
]

for col in money_columns:
    df_eligible_employees[col] = (
        df_eligible_employees[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .replace('nan', 0)
        .astype(float)
    )


Convert monetary columns


In [10]:
df_eligible_employees['Total_Deferrals'] = (
    df_eligible_employees['Employee_deduction_1_Pre-tax_amount'] +
    df_eligible_employees['Employee_deduction_2_Roth_Deferral']
)

df_eligible_employees['Capped_Compensation'] = (
    df_eligible_employees['Plan_eligible_comp']
    .clip(upper=305000)
)

df_eligible_employees['Deferral_Rate'] = (
    df_eligible_employees['Total_Deferrals'] /
    df_eligible_employees['Capped_Compensation']
)

df_eligible_employees[
    [
        'Plan_eligible_comp',
        'Capped_Compensation',
        'Total_Deferrals',
        'Deferral_Rate'
    ]
]

,Plan_eligible_comp,Capped_Compensation,Total_Deferrals,Deferral_Rate
0,82460.35,82460.35,17219.53,0.208822
2,319773.45,305000.00,8716.71,0.028579
3,60120.66,60120.66,0.00,0.000000
4,71936.25,71936.25,3925.84,0.054574
6,53881.01,53881.01,0.00,0.000000
8,36020.04,36020.04,1037.10,0.028792
9,53497.69,53497.69,0.00,0.000000
10,28326.44,28326.44,812.72,0.028691
11,98423.08,98423.08,2682.79,0.027258
12,93771.97,93771.97,3485.10,0.037166


In [11]:
def calculate_match_rate(deferral_rate):
    # 100% match on the first 3%
    if deferral_rate <= 0.03:
        return deferral_rate

    # 100% on the first 3% PLUS 50% on the next 2% (up to 5% total)
    elif deferral_rate <= 0.05:
        return 0.03 + ((deferral_rate - 0.03) * 0.5)

    # Cap at 4% total match (when deferral is 5% or more)
    else:
        return 0.04

df_eligible_employees['Match_Rate_Calculated'] = (
    df_eligible_employees['Deferral_Rate'].apply(calculate_match_rate)
)


# Calculate expected annual employer match
df_eligible_employees['Expected_Annual_Match'] = (
    df_eligible_employees['Capped_Compensation'] *
    df_eligible_employees['Match_Rate_Calculated']
)

In [12]:
# Finally, we calculate the True-up Amount for each eligible employee
df_eligible_employees['True-up'] = df_eligible_employees.apply(
    lambda row: max(
        (
            # 1. Apply the IRS compensation cap ($305,000 for 2022) to the base salary
            min(row['Plan_eligible_comp'], 305000) *

            # 2. Calculate the theoretical Match Rate based on the employee's contribution
            calculate_match_rate(
                (
                    # Sum of all personal contributions (Pre-tax + Roth)
                    row['Employee_deduction_1_Pre-tax_amount'] +
                    row['Employee_deduction_2_Roth_Deferral']
                ) /
                # Dividing by capped compensation to get the accurate contribution percentage
                min(row['Plan_eligible_comp'], 305000)
            )
        ) -
        # 3. Subtract the match amount already paid by the employer during the year
        row['Employer_Match'],

        # 4. Ensure the True-up is never negative (the company won't take money back)
        0
    ),
    # Process row by row (employee by employee)
    axis=1
)

In [13]:
df_eligible_employees['True-up'] = (
    df_eligible_employees['True-up']
    .round(2)
)

In [14]:
# 1. Define the currency columns again
currency_columns = [
    'Plan_eligible_comp',
    'Employee_deduction_1_Pre-tax_amount',
    'Employee_deduction_2_Roth_Deferral',
    'Employer_Match', # Note: verify if it is 'Employer_Match' or 'Employer_match_contribution'
    'Capped_Compensation',
    'Expected_Annual_Match',
    'Total_Deferrals',
    'True-up'
]

# 2. Create the display copy from your filtered data
final_output = df_eligible_employees.copy()

# 3. Apply the formatting to the float columns
for col in currency_columns:
    # We use a lambda to format the float as a currency string
    final_output[col] = final_output[col].apply(lambda x: f"${x:,.2f}")

# 4. Success! Now you can save it
print("Formatting complete. Ready to export.")
display(final_output.head())

Formatting complete. Ready to export.


,ID,Birth_Date,Hire_Date,Termination_Date,Rehire_Date,Gross_compensation,Plan_eligible_comp,Employee_deduction_1_Pre-tax_amount,Employee_deduction_2_Roth_Deferral,Employer_Match,Hours_worked_2022,Employee_Status,Entry_date,Total_Deferrals,Capped_Compensation,Deferral_Rate,Match_Rate_Calculated,Expected_Annual_Match,True-up
0,Employee 1,1944-05-14,2020-01-07,NaT,1970-01-01,82460.35,"$82,460.35","$17,219.53",$0.00,"$13,948.47",2080,Full-Time,2020-03-01,"$17,219.53","$82,460.35",0.208822,0.040000,"$3,298.41",$0.00
2,Employee 3,1948-08-28,2020-01-01,NaT,1970-01-01,319773.45,"$319,773.45","$8,716.71",$0.00,"$7,472.55",2080,Full-Time,2020-02-01,"$8,716.71","$305,000.00",0.028579,0.028579,"$8,716.71","$1,244.16"
3,Employee 4,1972-10-20,2013-11-18,NaT,1970-01-01,60120.66,"$60,120.66",$0.00,$0.00,$0.00,2148,Full-Time,2014-01-01,$0.00,"$60,120.66",0.000000,0.000000,$0.00,$0.00
4,Employee 5,1991-04-03,2017-06-07,2023-02-20,1970-01-01,71936.25,"$71,936.25","$3,925.84",$0.00,"$3,515.56",2080,Full-Time,2017-08-01,"$3,925.84","$71,936.25",0.054574,0.040000,"$2,877.45",$0.00
6,Employee 7,1993-04-03,2012-07-01,NaT,1970-01-01,53881.01,"$53,881.01",$0.00,$0.00,$0.00,2071,Full-Time,2012-08-01,$0.00,"$53,881.01",0.000000,0.000000,$0.00,$0.00


In [15]:
# Saving only the True-Up (Optional)

final_output['Rehire_Date'] = final_output['Rehire_Date'].replace('01/01/1970', 0)

final_true_up = final_output[['ID',
        'Birth_Date',
        'Hire_Date',
        'Termination_Date',
        'Rehire_Date',
        'Gross_compensation',
        'Plan_eligible_comp',
        'Employee_deduction_1_Pre-tax_amount',
        'Employee_deduction_2_Roth_Deferral',
        'Employer_Match',
        'Hours_worked_2022',
        'Employee_Status',
        'Entry_date',
        'True-up']].copy()

/tmp/ipykernel_3931/3857261548.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_output['Rehire_Date'] = final_output['Rehire_Date'].replace('01/01/1970', 0)


In [16]:
final_true_up.to_csv(
    'Final_TrueUp_Calculation_Report_2022_All_Info.csv',
    index=False
)